In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
)
from imblearn.over_sampling import SMOTE
import joblib
import matplotlib.pyplot as plt

# 1. Load Cleaned Dataset
df = pd.read_csv("titanic.csv")
df = df.drop(columns=['deck']) # drop deck (>30% missing)

# Define features and targets
X = df.drop(columns=['survived', 'alive'])
y = df['survived']

# Stratified Split FIRST to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessing Pipeline Setup
num_cols = ['age', 'fare', 'sibsp', 'parch']
cat_cols = ['sex', 'embarked', 'pclass', 'who']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# 2. Train Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=4),
    "Random Forest": RandomForestClassifier(random_state=42)
}

results = []

for name, clf in classifiers.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    # Fit ONLY on Train set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

# Render Decision Tree
dt_pipe = Pipeline([('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier(max_depth=3))])
dt_pipe.fit(X_train, y_train)
plt.figure(figsize=(12,8))
plot_tree(dt_pipe.named_steps['classifier'], filled=True)
plt.savefig("decision_tree.png")
plt.close()

# 3. Class Imbalance Comparison
# SMOTE on train fold only
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_proc, y_train)

clf_baseline = LogisticRegression().fit(X_train_proc, y_train)
clf_weighted = LogisticRegression(class_weight='balanced').fit(X_train_proc, y_train)
clf_smote = LogisticRegression().fit(X_train_smote, y_train_smote)

# 4. Hyperparameter Tuning on Random Forest (with oob_score=True)
rf_tune_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(oob_score=True, random_state=42))
])

param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5, None],
    'classifier__max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(rf_tune_pipe, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
oob_val = best_rf.named_steps['classifier'].oob_score_
print(f"Best RF Parameters: {grid_search.best_params_}")
print(f"Random Forest OOB Score: {oob_val:.4f}")

# 5. Regression Side-Task (Predicting Fare)
X_reg = df.drop(columns=['fare', 'alive'])
y_reg = df['fare'].fillna(df['fare'].median())

X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_num_cols = ['age', 'sibsp', 'parch']
reg_cat_cols = ['sex', 'embarked', 'pclass']

reg_preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), reg_num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), reg_cat_cols)
])

reg_pipe = Pipeline([('preprocessor', reg_preprocessor), ('model', LinearRegression())])
reg_pipe.fit(X_r_train, y_r_train)

y_r_pred = reg_pipe.predict(X_r_test)

mae = mean_absolute_error(y_r_test, y_r_pred)
rmse = np.sqrt(mean_squared_error(y_r_test, y_r_pred))
r2 = r2_score(y_r_test, y_r_pred)
adj_r2 = 1 - (1 - r2) * (len(y_r_test) - 1) / (len(y_r_test) - X_r_test.shape[1] - 1)

# Plot Residuals
residuals = y_r_test - y_r_pred
plt.figure(figsize=(6,4))
plt.scatter(y_r_pred, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residual Plot (Linear Regression on Fare)")
plt.xlabel("Predicted Fare")
plt.ylabel("Residuals")
plt.savefig("residuals.png")
plt.close()

# 6. Save Complete Pipeline Artifact
joblib.dump(best_rf, "best_model_pipeline.joblib")

# Reload verification
loaded_pipeline = joblib.load("best_model_pipeline.joblib")
sample_raw_input = pd.DataFrame([{
    'pclass': 1, 'sex': 'female', 'age': 29.0, 'sibsp': 0, 'parch': 0,
    'fare': 211.3375, 'embarked': 'S', 'class': 'First', 'who': 'woman',
    'adult_male': False, 'alone': True, 'embark_town': 'Southampton'
}])
print(f"Reloaded Pipeline Prediction on Raw Data: {loaded_pipeline.predict(sample_raw_input)}")
